In [ ]:
#promieniowanie kosmiczne (Oulu) oczyszczone z wpływu Słońca
#metodą fizyczną (Mishev i in. 2020; Vaisanen i in. 2023).



  #1. Czyści plik phi (format daty, usuwa braki).
  #2. Usuwa lata 1964-1974 z Oulu
  #3. Przycina oba zbiory do końca 2020 i wyrównuje po dacie.
  #4. Liczy całkę teoretycznego zliczenia N*(phi) (Mishev/Vaisanen).
  #5. Koryguje ją wolno zmienną kappa(t) (mediana krocząca ~4 lata).
  #6. Residuum = zmierzone dane - przeskalowana teoria.
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.integrate import quad

# ścieki
DESKTOP = Path.home() / "Desktop"
OULU_FILE = DESKTOP / "OuluCRpress+eff.csv"
PHI_RAW_FILE = DESKTOP / "phidziennie.csv"

OUTPUT_DIR = DESKTOP / "kosmosejsmiczne" / "bezsloncares"
OUTPUT_FILE = OUTPUT_DIR / "OuluResiduumBezSlonca.csv"

# parametry globalne
ROOF_SNOW_CUTOFF = "1975-01-01"   # dane Oulu przed tą datą odrzucone z powodu śniegu na dachu stacji i błędów w danych
END_CUTOFF = "2020-12-31"         # oba zbiory przycięte do tej daty (włącznie)

OULU_RC = 0.65      #  sztywność odcięcia geomagnetycznego - stała z artykułu Vaisanen i in. 2023
OULU_H = 1019.72    # g/cm^2, głębokość atmosferyczna - stała z artykułu Vaisanen i in. 2023

E_UPPER_LIMIT = 1000.0   #  górna granica całkowania - próg nieosiagalny dla czastek
PHI_GRID_MIN = 300.0     # - dolna granica siatki słuzacej do interpolacji N*(phi)
PHI_GRID_MAX = 2100.0    # - górna granica siatki słuzacej do interpolacji N*(phi)
PHI_GRID_STEP = 5.0      # gęstość siatki słuzacej do interpolacji N*(phi)

KAPPA_WINDOW_DAYS = 1460  # ~4 lata, mediana krocząca usuwająca wieloletni dryf

E0_CUTOFF = 0.938 #masa spoczynkowa protonu
TR_PROTON = 0.938 #energia spoczynkowa na jednej nukleon dla protonu
TR_ALPHA = 0.932 #energia spoczynkowa na jednej nukleon dla cząstki alfa
Z_A_RATIO = {"proton": 1.0, "alpha": 0.5} #stosunek liczby atomowej do masowej.
# wspolczynniki z artykulu Mishev i in. 2020,
_YIELD_1000_COEFFS = {
    "proton": [
        (0.0, 1.28, -12.104, 13.879, -8.6616, 0.0),
        (1.28, 10.0, -8.76, 2.831, 0.428, -0.186),
        (10.0, np.inf, -4.763, 1.206, -0.0365, 0.0),
    ],
    "alpha": [
        (0.0, 1.7, -9.5396, 7.2827, -3.0659, 0.5082),
        (1.7, 15.0, -8.65, 4.9329, -1.2022, 0.1179),
        (15.0, np.inf, -4.763, 1.206, -0.0365, 0.0),
    ],
}

_AB_COEFFS = {
    "proton": {
        "A": [-9.823e-07, 3.355e-06, -3.402e-06, 1.115e-06, -1.461e-07, 6.945e-09],
        "B": [1.186e-02, -4.713e-03, 2.348e-03, -6.394e-04, 8.091e-05, -3.963e-06],
    },
    "alpha": {
        "A": [-5.545e-06, 1.203e-05, -7.828e-06, 2.037e-06, -2.284e-07, 9.422e-09],
        "B": [1.458e-02, -1.017e-02, 5.176e-03, -1.226e-03, 1.316e-04, -5.351e-06],
    },
}


# czyszczenie pliku phi (usuwa braki, konwertuje datę)
def clean_phi(input_file: Path) -> pd.DataFrame:
    print(f"Czyszczenie phi: {input_file}")
    df = pd.read_csv(input_file, sep=";", encoding="utf-8-sig")
    df["Data"] = pd.to_datetime(df[["Year", "Month", "Day"]])
    before = len(df)
    df = df.dropna(subset=["Modulation potential [MV]"]).copy()
    print(f"  Usunięto {before - len(df)} dni z brakiem wartości phi.")
    df = df.rename(columns={"Modulation potential [MV]": "ModulationPotential_MV"})
    return df[["Data", "ModulationPotential_MV"]]


# wyrównanie danych Oulu i phi po dacie, przycięcie do końca 2020
def align_oulu_phi(oulu_file: Path, phi_df: pd.DataFrame) -> pd.DataFrame:
    print(f"Wczytywanie Oulu: {oulu_file}")
    oulu = pd.read_csv(oulu_file)
    oulu["Data"] = pd.to_datetime(oulu["Data"])

    before = len(oulu)
    oulu = oulu[oulu["Data"] >= pd.Timestamp(ROOF_SNOW_CUTOFF)].copy()
    print(f"  Usunięto {before - len(oulu)} dni Oulu sprzed {ROOF_SNOW_CUTOFF} (śnieg na dachu).")

    end_cutoff = pd.Timestamp(END_CUTOFF)
    oulu = oulu[oulu["Data"] <= end_cutoff].copy()
    phi_df = phi_df[phi_df["Data"] <= end_cutoff].copy()

    merged = pd.merge(oulu, phi_df, on="Data", how="inner")
    merged = merged.sort_values("Data").reset_index(drop=True)
    print(f"  Po wyrównaniu: {len(merged)} wspólnych dni "
          f"({merged['Data'].min().date()} -- {merged['Data'].max().date()})")
    return merged


# Funkcja wydajności (yield) dla cząstek protonów i alfa, zgodnie z Mishev i in. 2020
def _R_from_E(E):
    return np.sqrt(E * (E + 1.876))


def yield_1000(E, species):
    R = _R_from_E(E)
    lnR = np.log(R)
    for E_min, E_max, a0, a1, a2, a3 in _YIELD_1000_COEFFS[species]:
        if E_min < E <= E_max or (E_min == 0.0 and E <= E_max):
            return np.exp(a0 + a1 * lnR + a2 * lnR**2 + a3 * lnR**3)
    raise ValueError(f"Energia E={E} poza zdefiniowanymi zakresami dla {species}")


def yield_function(E, h, species):
    R = _R_from_E(E)
    lnR = np.log(R)
    b_A = _AB_COEFFS[species]["A"]
    b_B = _AB_COEFFS[species]["B"]
    A_val = sum(b * lnR**l for l, b in enumerate(b_A))
    B_val = sum(b * lnR**l for l, b in enumerate(b_B))
    return yield_1000(E, species) * np.exp(A_val * (1000 - h)**2 + B_val * (1000 - h))


# Funckja J_i z artykułu Vaisanen i in. 2023, czyli strumień cząstek modulowany potencjałem phi
def C_alpha(phi_MV):
    return 4.3e-9 * phi_MV**2 - 6.2e-7 * phi_MV + 0.337


def J_LIS(T, species):
    Tr = TR_PROTON if species == "proton" else TR_ALPHA
    beta = np.sqrt(T * (T + 2 * Tr)) / (T + Tr)
    return 2700.0 * T**1.12 / beta**2 * ((T + 0.67) / 1.67) ** (-3.93)


def J_modulated(T, phi_MV, species):
    Tr = TR_PROTON if species == "proton" else TR_ALPHA
    Z_A = Z_A_RATIO[species]
    Phi_i = Z_A * phi_MV / 1000.0

    T_shifted = T + Phi_i
    if T_shifted <= 0:
        return 0.0

    J_lis_shifted = J_LIS(T_shifted, species)
    if species == "alpha":
        J_lis_shifted *= C_alpha(phi_MV)

    numerator = T * (T + 2 * Tr)
    denominator = T_shifted * (T_shifted + 2 * Tr)
    return J_lis_shifted * numerator / denominator


# Obliczenie całki
def energy_threshold(Rc, species):
    Z_A = Z_A_RATIO[species]
    return np.sqrt((Z_A * Rc) ** 2 + E0_CUTOFF ** 2) - E0_CUTOFF


def N_star(phi_MV, Rc, h):
    total = 0.0
    for species in ["proton", "alpha"]:
        Ec = energy_threshold(Rc, species)

        def integrand(E, species=species):
            return yield_function(E, h, species) * J_modulated(E, phi_MV, species)

        value, _ = quad(integrand, Ec, E_UPPER_LIMIT, limit=200)
        total += value
    return total


# Obliczenie residuum: surowe dane Oulu - przeskalowana teoria N*(phi)
def compute_residuum(df: pd.DataFrame) -> pd.DataFrame:
    print(f"Liczenie N*(phi) na siatce {PHI_GRID_MIN}-{PHI_GRID_MAX} MV...")
    phi_grid = np.arange(PHI_GRID_MIN, PHI_GRID_MAX + PHI_GRID_STEP, PHI_GRID_STEP)
    n_star_grid = np.array([N_star(phi, OULU_RC, OULU_H) for phi in phi_grid])
    assert np.all(np.diff(n_star_grid) < 0), "N* powinno maleć z rosnącym phi!"

    if df["ModulationPotential_MV"].min() < PHI_GRID_MIN or \
       df["ModulationPotential_MV"].max() > PHI_GRID_MAX:
        print("  UWAGA: phi w danych wykracza poza siatkę -- rozważ jej poszerzenie.")

    df["N_star"] = np.interp(df["ModulationPotential_MV"], phi_grid, n_star_grid)
    df["kappa"] = df["N_star"] / df["CorrectedCountRate"]

    df["kappa_smooth"] = (
        df["kappa"].rolling(window=KAPPA_WINDOW_DAYS, center=True,
                             min_periods=KAPPA_WINDOW_DAYS // 4)
        .median()
        .bfill().ffill()
    )

    df["N_star_scaled"] = df["N_star"] / df["kappa_smooth"]
    df["Residuum"] = df["CorrectedCountRate"] - df["N_star_scaled"]

    redukcja = 100 * (1 - df["Residuum"].std() / df["CorrectedCountRate"].std())
    print(f"Std surowe: {df['CorrectedCountRate'].std():.2f}, "
          f"std residuum: {df['Residuum'].std():.2f} (redukcja {redukcja:.1f}%)")

    return df[["Data", "CorrectedCountRate", "ModulationPotential_MV",
               "N_star", "kappa", "kappa_smooth", "N_star_scaled", "Residuum"]]


# uruchomienie
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    phi_df = clean_phi(PHI_RAW_FILE)
    merged = align_oulu_phi(OULU_FILE, phi_df)
    result = compute_residuum(merged)

    result_out = result.copy()
    result_out["Data"] = result_out["Data"].dt.strftime("%Y-%m-%d")
    result_out.to_csv(OUTPUT_FILE, index=False)
    print(f"\nZapisano {len(result_out)} wierszy do: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()